# The exposure dip test (plus everything from the long-training notebook)

**Why this notebook.** In the last long run, DINOv2 trained by *exposure* briefly pulled abstract and representational art **together** (split ratio 1.01 → 0.76 at epoch 2) before drifting back up to 1.10. That would be the expert-like direction, but it happened at the same moment the model jumped abruptly away from its starting point, which is typical of a freshly added, randomly initialised training head sending large, unhelpful updates into the model. This notebook tests whether the dip is real or an artefact.

**The test.** Train DINOv2 by exposure with and without a **head warm-up** (the new head is first trained on its own, with the model frozen, so it can't disturb the features), with **3 seeds** each, recording **every epoch** early on.
- Dip appears without warm-up but disappears with it → artefact of the random head.
- Dip survives with warm-up, across seeds → exposure genuinely pulls the two kinds of art together early on.

**Modifications compared with the previous long-training notebook (v2):**
1. **Head warm-up** (`WARMUPS`): optional training of only the new head, with the model frozen, before normal training. Runs with different warm-ups are compared automatically in one go.
2. **Denser early recording:** every epoch for epochs 1–6, then every 2 epochs.
3. **Full precision for DINOv3** (`AMP_OFF_FOR`): fixes the NaN loss; DINOv3's large internal values overflow 16-bit numbers. DINOv2 keeps the faster mixed precision.
4. **Gradient clipping** (`GRAD_CLIP`): limits sudden large updates; aimed at the loss spikes seen late in DINOv2 label training.
5. **Focused defaults:** DINOv2, exposure only, 3 seeds, 12 epochs, warm-up 0 vs 3 epochs. About 15–20 minutes on a T4.
6. **New section 0, "The dip test":** early split ratio for each warm-up and seed, with a verdict.
7. **Figures generalised:** each run gets its own colour (solid = labels, dashed = exposure), so runs that differ only by warm-up can be told apart.

**Unchanged from v2:** same paintings and splits, same learning rate (3e-5) and blocks trained (last 4), same exposure objective and augmentations, and all analyses (learning curves, matched change, maps over training, vector-vs-field breakdown, abstract-vs-representational movement, arrow field, labels-vs-exposure directions, local spatial structure).

**For the full study again:** set `MODELS` to both models, `OBJECTIVES = ["labels", "exposure"]`, `EPOCHS = 30`, and `WARMUPS` to the warm-up you settle on.

**Before running:** T4 GPU (High-RAM helps); `HF_TOKEN` secret only if you add DINOv3.

In [ ]:
!pip -q install datasets umap-learn "transformers>=4.56"

In [ ]:
import os
os.environ["USE_TF"] = "0"

import copy, gc, math, random, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms as T
from datasets import load_dataset
from transformers import AutoModel
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import umap

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("WARNING: no GPU attached. Switch to a T4 GPU runtime.")

**Kaggle only:** uncomment to load your Hugging Face token.

In [ ]:
# from kaggle_secrets import UserSecretsClient
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

## Settings

In [ ]:
MODELS = ["facebook/dinov2-small"]          # add "facebook/dinov3-vits16-pretrain-lvd1689m" for the full study
OBJECTIVES = ["exposure"]                   # ["labels", "exposure"] for the full study
WARMUPS = [0, 3]                            # head warm-up epochs to compare (model frozen during warm-up)

LR = 3e-5                 # same for both models and both objectives
N_BLOCKS = 4
EPOCHS = 12               # 30 for the full study
RECORD_EVERY = 2          # after the dense early phase, measure every N epochs
DENSE_UNTIL = 6           # measure every epoch up to here
BATCH = {"labels": 32, "exposure": 64}
SSL_TEMPERATURE = 0.1
USE_AMP = True            # mixed precision: ~2x faster on a T4
AMP_OFF_FOR = ["dinov3"]  # models that need full precision (DINOv3 overflows 16-bit numbers → NaN loss)
GRAD_CLIP = 1.0           # limit sudden large updates (None to switch off)
SEEDS = [0, 1, 2]

STYLES = ["Impressionism", "Baroque", "Cubism", "Ukiyo_e", "Pop_Art",
          "Abstract_Expressionism", "Realism", "Color_Field_Painting"]
PER_STYLE = 120
MAX_SCAN = 25000
ABSTRACT = ["Abstract_Expressionism", "Color_Field_Painting"]
REPRESENTATIONAL = ["Baroque", "Realism", "Impressionism", "Ukiyo_e"]

QUERY = (0.35, 0.5)       # spatial maps: query patch position as (row, column) fractions of the image
RECORD_EPOCHS = sorted(set([0] + list(range(1, min(DENSE_UNTIL, EPOCHS) + 1))
                           + list(range(RECORD_EVERY, EPOCHS + 1, RECORD_EVERY)) + [EPOCHS]))
print("Recording at epochs:", RECORD_EPOCHS)

## Load paintings

Fit (64%) for training, validation (16%) for monitoring, test (20%) for the probe.

In [ ]:
torch.manual_seed(0); np.random.seed(0); random.seed(0)
ds = load_dataset("huggan/wikiart", split="train", streaming=True)
if ds.features is None:
    ds = ds._resolve_features()
all_styles = ds.features["style"].names
style_ids = {all_styles.index(s): s for s in STYLES if s in all_styles}
id_to_label = {sid: i for i, sid in enumerate(style_ids)}
class_names = [style_ids[sid] for sid in style_ids]
K = len(class_names)

counts = {sid: 0 for sid in style_ids}
images, labels = [], []
for i, ex in enumerate(ds.shuffle(seed=0, buffer_size=1000)):
    sid = ex["style"]
    if sid in counts and counts[sid] < PER_STYLE:
        img = ex["image"].convert("RGB")
        img.thumbnail((320, 320))
        images.append(img); labels.append(id_to_label[sid]); counts[sid] += 1
    if all(c >= PER_STYLE for c in counts.values()) or i >= MAX_SCAN:
        break
labels = np.array(labels)
del ds; gc.collect()

tr_idx, te_idx = train_test_split(np.arange(len(images)), test_size=0.2, stratify=labels, random_state=0)
fit_idx, val_idx = train_test_split(tr_idx, test_size=0.2, stratify=labels[tr_idx], random_state=0)
ia_all = np.where(np.isin(labels, [class_names.index(n) for n in ABSTRACT if n in class_names]))[0]
ir_all = np.where(np.isin(labels, [class_names.index(n) for n in REPRESENTATIONAL if n in class_names]))[0]
test_imgs = [images[j] for j in te_idx]
EX_POS = [int(np.where(labels[te_idx] == k)[0][0]) for k in range(min(3, K))]   # example paintings for the maps
print(f"Kept {len(images)} paintings: {len(fit_idx)} fit / {len(val_idx)} validation / {len(te_idx)} test")

## Helpers

In [ ]:
def short(model_id):
    return model_id.split("/")[-1].replace("-pretrain-lvd1689m", "")

def probe_on(emb, train_ix, eval_ix):
    sc = StandardScaler().fit(emb[train_ix])
    clf = LogisticRegression(max_iter=3000).fit(sc.transform(emb[train_ix]), labels[train_ix])
    return clf.score(sc.transform(emb[eval_ix]), labels[eval_ix])

probe = lambda emb: probe_on(emb, tr_idx, te_idx)
val_probe = lambda emb: probe_on(emb, fit_idx, val_idx)

def split_ratio(emb, ia, ir):
    e = emb / np.linalg.norm(emb, axis=1, keepdims=True)
    A, R_ = e[ia], e[ir]
    ca, cr = A.mean(0), R_.mean(0)
    between = 1 - ca @ cr / (np.linalg.norm(ca) * np.linalg.norm(cr))
    within = np.mean([np.mean(1 - X @ c / np.linalg.norm(c)) for X, c in ((A, ca), (R_, cr))])
    return between / within

def linear_cka(X, Y):
    X = X.astype(np.float64) - X.mean(0); Y = Y.astype(np.float64) - Y.mean(0)
    return np.linalg.norm(Y.T @ X) ** 2 / (np.linalg.norm(X.T @ X) * np.linalg.norm(Y.T @ Y))

def show_img(img):
    """Same framing the models see: shorter side to 256, centre crop 224."""
    w, h = img.size
    sc = 256 / min(w, h)
    img = img.resize((max(256, round(w * sc)), max(256, round(h * sc))), Image.BICUBIC)
    w, h = img.size
    l, t = (w - 224) // 2, (h - 224) // 2
    return img.crop((l, t, l + 224, t + 224))

def upsample(a, size=224):
    a = np.asarray(Image.fromarray(a.astype(np.float32)).resize((size, size), Image.BILINEAR))
    return (a - a.min()) / (a.max() - a.min() + 1e-8)

def unit_rows(E):
    E = E.astype(np.float64)
    return E / np.linalg.norm(E, axis=1, keepdims=True)

def procrustes_rotation(B, A):
    """Rotation R (orthogonal) that best maps B onto A, fitted on centred data."""
    U, _, Vt = np.linalg.svd((B - B.mean(0)).T @ (A - A.mean(0)))
    return U @ Vt

def field_stats(E0, Et, align=True):
    """Split how paintings moved (E0 -> Et, unit-normalised) into:
       shared   = one common shift for all paintings (the steering *vector*),
       style    = extra movement shared within each style (a style-organised *field*),
       other    = movement specific to individual paintings.
    Fractions of total squared movement; they add to 1."""
    A, B = unit_rows(E0), unit_rows(Et)
    if align:
        B = B @ procrustes_rotation(B, A)       # remove any pure rotation of the space
    D = B - A
    total = (D ** 2).sum(1).mean()
    mean_shift = D.mean(0)
    shared = (mean_shift ** 2).sum()
    Dr = D - mean_shift
    style = sum((labels == k).sum() * (Dr[labels == k].mean(0) ** 2).sum() for k in range(K)) / len(labels)
    u = A[ia_all].mean(0) - A[ir_all].mean(0); u /= np.linalg.norm(u)
    proj = Dr @ u                                # + = towards the abstract side of the original axis
    return dict(total=total, shared=shared / total, style=style / total, other=1 - (shared + style) / total,
                abstract_move=proj[ia_all].mean(), representational_move=proj[ir_all].mean(), D=D, Dr=Dr, B=B, A=A)

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
eval_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)])
label_tf = T.Compose([T.RandomResizedCrop(224, scale=(0.6, 1.0)), T.RandomHorizontalFlip(),
                      T.ToTensor(), T.Normalize(MEAN, STD)])
ssl_tf = T.Compose([T.RandomResizedCrop(224, scale=(0.3, 1.0)), T.RandomHorizontalFlip(),
                    T.RandomApply([T.ColorJitter(brightness=0.2, contrast=0.2)], p=0.5),
                    T.ToTensor(), T.Normalize(MEAN, STD)])
AMP = USE_AMP and device == "cuda"

def load_backbone(model_id):
    return AutoModel.from_pretrained(model_id, attn_implementation="eager").to(device).eval()

def cls_of(model, x):
    return model(pixel_values=x).last_hidden_state[:, 0]

@torch.no_grad()
def embed(model, imgs, bs=64):
    model.eval()
    return torch.cat([cls_of(model, torch.stack([eval_tf(im) for im in imgs[i:i + bs]]).to(device)).float().cpu()
                      for i in range(0, len(imgs), bs)]).numpy()

def get_blocks(bb):
    for path in ("encoder.layer", "model.layer", "layer"):
        obj = bb
        try:
            for name in path.split("."):
                obj = getattr(obj, name)
        except AttributeError:
            continue
        if isinstance(obj, nn.ModuleList):
            return obj
    raise ValueError("Could not find the transformer blocks")

def get_final_norm(bb):
    return bb.layernorm if hasattr(bb, "layernorm") else bb.norm

def make_head(objective, hidden):
    if objective == "labels":
        return nn.Linear(hidden, K)
    return nn.Sequential(nn.Linear(hidden, 512), nn.GELU(), nn.Linear(512, 128))

def nt_xent(z, tau):
    z = F.normalize(z.float(), dim=1)            # computed in float32 for stability
    n = z.shape[0] // 2
    sim = z @ z.T / tau - 1e9 * torch.eye(2 * n, device=z.device)
    target = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)
    return F.cross_entropy(sim, target)

@torch.no_grad()
def head_accuracy(model, head, index, bs=64):
    model.eval(); head.eval()
    correct = 0
    for i in range(0, len(index), bs):
        b = index[i:i + bs]
        x = torch.stack([eval_tf(images[j]) for j in b]).to(device)
        correct += int((head(cls_of(model, x)).argmax(1).cpu().numpy() == labels[b]).sum())
    return correct / len(index)

@torch.no_grad()
def trainable_change(base, model):
    pb = dict(base.named_parameters())
    d = n = 0.0
    for name, p in model.named_parameters():
        if p.requires_grad:
            d += (p - pb[name]).float().pow(2).sum().item(); n += pb[name].float().pow(2).sum().item()
    return math.sqrt(d) / (math.sqrt(n) + 1e-12)

@torch.no_grad()
def patch_sims(model, imgs, bs=32):
    """Per painting: cosine similarity between every pair of final-layer patch tokens. (images, P, P), float16 on GPU."""
    model.eval()
    out = []
    for i in range(0, len(imgs), bs):
        px = torch.stack([eval_tf(im) for im in imgs[i:i + bs]]).to(device)
        h = model(pixel_values=px).last_hidden_state[:, 1:]
        side = int(math.sqrt(h.shape[1]))
        h = F.normalize(h[:, -side * side:].float(), dim=-1)     # patch tokens only (drops register tokens)
        out.append((h @ h.transpose(1, 2)).half())
    return torch.cat(out)

_MASKS = {}
def grid_masks(side, dev):
    if side not in _MASKS:
        yy, xx = torch.meshgrid(torch.arange(side), torch.arange(side), indexing="ij")
        c = torch.stack([yy.flatten(), xx.flatten()], 1).float()
        d = torch.cdist(c, c)
        near = (d > 0) & (d <= 1.5)                                # the 8 neighbouring patches
        far = d >= side / 2                                        # patches at least half the image apart
        triu = torch.triu(torch.ones_like(d, dtype=torch.bool), diagonal=1)
        _MASKS[side] = (near.to(dev), far.to(dev), triu.to(dev))
    return _MASKS[side]

@torch.no_grad()
def spatial_metrics(S, S0):
    """local: neighbours minus distant similarity (higher = more spatially organised);
       homog: mean similarity of all patch pairs (higher = patches more alike, global info swamping local);
       preserve: correlation of each painting's patch-similarity pattern with the untrained model's (1 = unchanged)."""
    near, far, triu = grid_masks(int(math.sqrt(S.shape[-1])), S.device)
    S, S0 = S.float(), S0.float()
    local = S[:, near].mean(1) - S[:, far].mean(1)
    a, b = S[:, triu], S0[:, triu]
    homog = a.mean(1)
    a = a - a.mean(1, keepdim=True); b = b - b.mean(1, keepdim=True)
    preserve = (a * b).sum(1) / (a.norm(dim=1) * b.norm(dim=1) + 1e-8)
    return dict(spatial_local=local.mean().item(), spatial_homog=homog.mean().item(),
                spatial_preserve=preserve.mean().item())

def train_long(base, objective, seed, on_record, warmup=0):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = copy.deepcopy(base).to(device)
    for p in model.parameters():
        p.requires_grad = False
    blocks = get_blocks(model)
    for block in blocks[max(len(blocks) - N_BLOCKS, 0):]:
        for p in block.parameters():
            p.requires_grad = True
    for p in get_final_norm(model).parameters():
        p.requires_grad = True
    head = make_head(objective, model.config.hidden_size).to(device)
    bs = BATCH[objective]

    # Head warm-up: train only the new head on the frozen model, so random head weights can't disturb the features
    head_opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=0.05)
    for _ in range(warmup):
        model.eval(); head.train()
        order = np.random.permutation(fit_idx)
        for i in range(0, len(order), bs):
            b = order[i:i + bs]
            if len(b) < 2:
                continue
            with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
                if objective == "labels":
                    x = torch.stack([label_tf(images[j]) for j in b]).to(device)
                else:
                    x = torch.stack([ssl_tf(images[j]) for j in b] + [ssl_tf(images[j]) for j in b]).to(device)
                feats = cls_of(model, x).float()
            if objective == "labels":
                loss = F.cross_entropy(head(feats), torch.tensor(labels[b]).to(device))
            else:
                loss = nt_xent(head(feats), SSL_TEMPERATURE)
            head_opt.zero_grad(set_to_none=True); loss.backward(); head_opt.step()
    del head_opt

    opt = torch.optim.AdamW([
        {"params": [p for p in model.parameters() if p.requires_grad], "lr": LR},
        {"params": head.parameters(), "lr": 1e-3}], weight_decay=0.05)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    trainable = [p for p in model.parameters() if p.requires_grad] + list(head.parameters())
    for epoch in range(1, EPOCHS + 1):
        model.train(); head.train()
        order, losses = np.random.permutation(fit_idx), []
        for i in range(0, len(order), bs):
            b = order[i:i + bs]
            if len(b) < 2:
                continue
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
                if objective == "labels":
                    x = torch.stack([label_tf(images[j]) for j in b]).to(device)
                    loss = F.cross_entropy(head(cls_of(model, x)).float(), torch.tensor(labels[b]).to(device))
                else:
                    x = torch.stack([ssl_tf(images[j]) for j in b] + [ssl_tf(images[j]) for j in b]).to(device)
                    loss = nt_xent(head(cls_of(model, x)), SSL_TEMPERATURE)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if GRAD_CLIP:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
            scaler.step(opt); scaler.update()
            losses.append(loss.item())
        if epoch in RECORD_EPOCHS:
            on_record(epoch, model, head, float(np.mean(losses)))
    del model, head, opt

## Run

For each model: measure the untrained version, then train each objective for `EPOCHS`, measuring every `RECORD_EVERY` epochs. Embeddings are kept at every record point for the later analyses.

In [ ]:
R = {"before": {}, "runs": {}}
t0 = time.time()
clock = lambda: f"[{(time.time() - t0) / 60:.1f} min]"

for model_id in MODELS:
    AMP = USE_AMP and device == "cuda" and not any(k in model_id for k in AMP_OFF_FOR)
    print(f"\n=== {short(model_id)} (mixed precision: {AMP}) ===")
    try:
        base = load_backbone(model_id)
    except Exception as err:
        print(f"  could not load, skipping: {type(err).__name__}: {str(err)[:200]}"); continue
    emb0 = embed(base, images)
    R["before"][model_id] = dict(emb=emb0, probe=probe(emb0), val_probe=val_probe(emb0),
                                 sil=silhouette_score(emb0, labels, metric="cosine"),
                                 split=split_ratio(emb0, ia_all, ir_all))
    S0 = patch_sims(base, test_imgs)
    R["before"][model_id].update(spatial=spatial_metrics(S0, S0), S_examples=S0[EX_POS].float().cpu().numpy())
    B = R["before"][model_id]
    print(f"  before: local structure {B['spatial']['spatial_local']:.3f}, patch sameness {B['spatial']['spatial_homog']:.3f}")
    print(f"  before: probe {B['probe']:.3f}, silhouette {B['sil']:.3f}, split {B['split']:.3f} {clock()}")

    for objective in OBJECTIVES:
      for warmup in WARMUPS:
        for seed in SEEDS:
            name = f"{short(model_id)} · {objective} · warm-up {warmup}"
            run = dict(base=model_id, objective=objective, warmup=warmup, seed=seed, embs={0: emb0.astype(np.float16)},
                       traj=[dict(epoch=0, silhouette=B["sil"], split_ratio=B["split"], val_probe=B["val_probe"],
                                  cka_to_base=1.0, weight_change=0.0, loss=np.nan, val_acc=np.nan, **B["spatial"])])
            def on_record(ep, m, h, loss):
                if not np.isfinite(loss):
                    raise RuntimeError(f"{name}: loss is {loss} at epoch {ep}; training broke (add the model to AMP_OFF_FOR)")
                e = embed(m, images)
                run["embs"][ep] = e.astype(np.float16)
                rec = dict(epoch=ep, silhouette=silhouette_score(e, labels, metric="cosine"),
                           split_ratio=split_ratio(e, ia_all, ir_all), val_probe=val_probe(e),
                           cka_to_base=linear_cka(emb0, e), weight_change=trainable_change(base, m), loss=loss,
                           val_acc=head_accuracy(m, h, val_idx) if objective == "labels" else np.nan)
                S = patch_sims(m, test_imgs)
                rec.update(spatial_metrics(S, S0))
                if ep == EPOCHS:
                    run["S_examples"] = S[EX_POS].float().cpu().numpy()
                del S
                run["traj"].append(rec)
                print(f"  {name} s{seed} ep {ep:>2}: loss {loss:.3f}, silhouette {rec['silhouette']:.3f}, "
                      f"split {rec['split_ratio']:.3f}, val probe {rec['val_probe']:.3f}, "
                      f"CKA {rec['cka_to_base']:.3f}, local structure kept {rec['spatial_preserve']:.3f} {clock()}")
            train_long(base, objective, seed, on_record, warmup=warmup)
            last = run["embs"][max(run["embs"])].astype(np.float32)
            run["probe_final"] = probe(last)
            R["runs"][(name, seed)] = run
            gc.collect(); torch.cuda.empty_cache()
    del base, S0; gc.collect(); torch.cuda.empty_cache()
print(f"\nDone {clock()}")

## Figures setup

In [ ]:
if not R["runs"]:
    raise RuntimeError("No run finished; check the messages above.")
BASES = list(R["before"])
OBJ_LINE = {"labels": "-", "exposure": "--"}
STYLE_CMAP = plt.get_cmap("tab10")
MAIN = {name: run for (name, seed), run in R["runs"].items() if seed == SEEDS[0]}
RUN_COL = {name: plt.get_cmap("tab10")(i) for i, name in enumerate(MAIN)}
TRAJ = {key: pd.DataFrame(run["traj"]) for key, run in R["runs"].items()}

summary = []
for (name, seed), run in R["runs"].items():
    t = TRAJ[(name, seed)].iloc[-1]; B = R["before"][run["base"]]
    summary.append(dict(run=name, seed=seed, probe_before=B["probe"], probe_after=run["probe_final"],
                        sil_before=B["sil"], sil_after=t.silhouette, split_before=B["split"], split_after=t.split_ratio,
                        cka_to_base=t.cka_to_base, weight_change=t.weight_change,
                        local_before=B["spatial"]["spatial_local"], local_after=t.spatial_local,
                        local_structure_kept=t.spatial_preserve))
print(pd.DataFrame(summary).set_index(["run", "seed"]).round(3).to_string())

## 0. The dip test

Split ratio over the first epochs for every warm-up and seed (thin lines = individual seeds, thick = mean), with the untrained value as a dotted line. Below 1 on the left axis would mean abstract and representational art moved *closer* than before training. The right panel shows how abruptly the model moved away from its starting point (similarity to the untrained model): the warm-up should remove the sudden early drop.

The table gives, per run, the lowest split ratio reached in epochs 1–`DENSE_UNTIL` and the drop from the untrained value. A drop larger than about 0.05 (roughly twice the bootstrap uncertainty of the split ratio seen earlier) counts as a dip.

In [ ]:
early = [e for e in RECORD_EPOCHS if e <= max(DENSE_UNTIL + 4, 10)]
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
rows = []
for name in MAIN:
    keys = [(n, s) for (n, s) in R["runs"] if n == name]
    split_mat = np.array([TRAJ[k].set_index("epoch").loc[early, "split_ratio"].values for k in keys])
    cka_mat = np.array([TRAJ[k].set_index("epoch").loc[early, "cka_to_base"].values for k in keys])
    for ax, mat in ((axes[0], split_mat), (axes[1], cka_mat)):
        for line in mat:
            ax.plot(early, line, color=RUN_COL[name], alpha=0.3, lw=1)
        ax.plot(early, mat.mean(0), color=RUN_COL[name], lw=2.5, marker="o", markersize=4, label=name)
    before = R["before"][MAIN[name]["base"]]["split"]
    for (n, s), line in zip(keys, split_mat):
        window = [v for e, v in zip(early, line) if 1 <= e <= DENSE_UNTIL]
        rows.append(dict(run=n, seed=s, before=before, lowest_early=min(window), drop=before - min(window)))
for b, B in R["before"].items():
    axes[0].axhline(B["split"], color="k", ls=":", lw=1)
axes[0].set_title("Abstract vs representational (split ratio), early epochs"); axes[0].set_ylabel("split ratio")
axes[1].set_title("Similarity to untrained model (CKA), early epochs")
for ax in axes:
    ax.set_xlabel("epoch"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

dip = pd.DataFrame(rows)
print(dip.set_index(["run", "seed"]).round(3).to_string())
print()
for name, g in dip.groupby("run", sort=False):
    n_dip = int((g["drop"] > 0.05).sum())
    print(f"{name}: mean drop {g['drop'].mean():+.3f} ± {g['drop'].std():.3f}, dip in {n_dip} of {len(g)} seeds")

## 1. Long learning curves

Colour = run, solid = labels, dashed = exposure. Faint lines are extra seeds. **Loss** is shown relative to the first recorded epoch, because the two objectives have different scales.

In [ ]:
panels = [("silhouette", "Style cluster tightness (silhouette)"), ("split_ratio", "Abstract vs representational (split ratio)"),
          ("val_probe", "Linear probe accuracy (validation)"), ("cka_to_base", "Similarity to untrained model (CKA)"),
          ("loss", "Training loss (relative to first record)"), ("val_acc", "Style classifier accuracy (labels only)")]
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for (name, seed), run in R["runs"].items():
    df = TRAJ[(name, seed)]
    kw = dict(color=RUN_COL[name], linestyle=OBJ_LINE[run["objective"]],
              alpha=1.0 if seed == SEEDS[0] else 0.3, label=name if seed == SEEDS[0] else None)
    for ax, (key, title) in zip(axes.ravel(), panels):
        y = df[key].values.astype(float)
        if key == "loss":
            first = y[~np.isnan(y)][0] if np.any(~np.isnan(y)) else 1.0
            y = y / first
        ax.plot(df.epoch, y, marker="o", markersize=3, **kw)
for ax, (key, title) in zip(axes.ravel(), panels):
    ax.set_title(title, fontsize=10); ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
axes[0, 2].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 2. Same amount of change, different direction?

Each objective plotted against how far the model has moved from the untrained one (1 − CKA). At equal distance travelled, do labels and exposure reorganise the space differently? This removes the "exposure was just trained less" explanation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.6))
for (name, seed), run in R["runs"].items():
    if seed != SEEDS[0]:
        continue
    df = TRAJ[(name, seed)]
    x = 1 - df.cka_to_base
    kw = dict(color=RUN_COL[name], linestyle=OBJ_LINE[run["objective"]], marker="o", markersize=3, label=name)
    axes[0].plot(x, df.split_ratio, **kw); axes[1].plot(x, df.silhouette, **kw); axes[2].plot(x, df.val_probe, **kw)
for ax, t in zip(axes, ["Split ratio", "Silhouette", "Linear probe accuracy (validation)"]):
    ax.set_title(f"{t} vs amount of change", fontsize=10); ax.set_xlabel("change from untrained (1 − CKA)"); ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 3. How the map changes over training

Embedding maps at four points in training, one row per model and objective.

In [ ]:
snap_targets = [0, EPOCHS // 6, EPOCHS // 2, EPOCHS]
umap_kw = dict(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=0)
rows = list(MAIN.items())
fig, axes = plt.subplots(len(rows), len(snap_targets), figsize=(4.2 * len(snap_targets), 3.9 * len(rows)), squeeze=False)
for i, (name, run) in enumerate(rows):
    recorded = sorted(run["embs"])
    for j, target in enumerate(snap_targets):
        ep = min(recorded, key=lambda e: abs(e - target))
        e = run["embs"][ep].astype(np.float32)
        xy = umap.UMAP(**umap_kw).fit_transform(e)
        ax = axes[i, j]
        for k, cname in enumerate(class_names):
            m = labels == k
            ax.scatter(xy[m, 0], xy[m, 1], s=4, color=STYLE_CMAP(k), label=cname, alpha=0.8)
        ax.set_title(f"{name}, epoch {ep}\nsplit {split_ratio(e, ia_all, ir_all):.2f}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
handles, labs = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labs, loc="center right", markerscale=3, fontsize=9)
plt.tight_layout(rect=(0, 0, 0.87, 1)); plt.show()

## 4. Steering vector or steering field?

For every recorded epoch, how each painting moved from the untrained model is split into three parts that add up to 100%:

- **shared:** one common shift for all paintings (a steering **vector**);
- **style:** extra movement shared within each style (a **field** organised by style);
- **individual:** movement specific to single paintings.

Embeddings are unit-normalised and rotationally aligned to the untrained ones first, so a pure rotation of the space isn't counted as movement.

In [ ]:
FIELD = {}
for name, run in MAIN.items():
    E0 = R["before"][run["base"]]["emb"]
    FIELD[name] = pd.DataFrame([dict(epoch=ep, **{k: v for k, v in field_stats(E0, run["embs"][ep].astype(np.float32)).items()
                                                 if k not in ("D", "Dr", "B", "A")})
                                for ep in sorted(run["embs"]) if ep > 0])

fig, axes = plt.subplots(1, len(FIELD), figsize=(4.6 * len(FIELD), 4.2), squeeze=False, sharey=True)
for ax, (name, df) in zip(axes[0], FIELD.items()):
    ax.stackplot(df["epoch"], df["shared"], df["style"], df["other"], labels=["shared (vector)", "style (field)", "individual"],
                 colors=["#9ecae1", "#fd8d3c", "#d9d9d9"])
    ax.set_title(name, fontsize=10); ax.set_xlabel("epoch"); ax.set_ylim(0, 1)
axes[0, 0].set_ylabel("fraction of total movement"); axes[0, -1].legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

print("Final epoch:")
for name, df in FIELD.items():
    t = df.iloc[-1]
    print(f"  {name}: shared {t['shared']:.0%}, style {t['style']:.0%}, individual {t['other']:.0%}, "
          f"total movement {t['total']:.4f}")

## 5. Which way do abstract and representational paintings move?

Movement along the original abstract–representational axis (after removing the shared shift). Solid = abstract paintings, dotted = representational; colour = run. If abstract paintings move up and representational ones down, training pulls the two groups apart; the reverse would pull them together (expert-like).

In [ ]:
fig, axes = plt.subplots(1, len(BASES), figsize=(7 * len(BASES), 4.4), squeeze=False)
for ax, b in zip(axes[0], BASES):
    for name, df in FIELD.items():
        if MAIN[name]["base"] != b:
            continue
        short_name = name.replace(f"{short(b)} · ", "")
        ax.plot(df["epoch"], df["abstract_move"], color=RUN_COL[name], marker="o", markersize=3, label=f"{short_name}: abstract")
        ax.plot(df["epoch"], df["representational_move"], color=RUN_COL[name], marker="s", markersize=3, ls=":",
                label=f"{short_name}: representational")
    ax.axhline(0, color="k", lw=0.8)
    ax.set_title(short(b), fontsize=11); ax.set_xlabel("epoch"); ax.set_ylabel("movement along abstract → axis")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 6. The field, drawn

Each arrow is one painting's movement from the untrained model to the end of training, projected onto the untrained model's two main axes and coloured by style (120 paintings, 15 per style). Arrow lengths are magnified by the same factor in every panel. Parallel arrows = vector; arrows that differ by style = field.

In [ ]:
rng = np.random.default_rng(0)
pick = np.concatenate([rng.choice(np.where(labels == k)[0], 15, replace=False) for k in range(K)])
proj = {}
for name, run in MAIN.items():
    E0 = R["before"][run["base"]]["emb"]
    fs = field_stats(E0, run["embs"][max(run["embs"])].astype(np.float32))
    pca = PCA(2).fit(fs["A"])
    p0, p1 = pca.transform(fs["A"]), pca.transform(fs["B"])
    proj[name] = (p0, p1 - p0)
med = max(np.median(np.linalg.norm(d[pick], axis=1)) for _, d in proj.values())
span = max(np.ptp(p0[:, 0]) for p0, _ in proj.values())
scale = 0.06 * span / (med + 1e-12)

names = list(proj)
cols = min(2, len(names)); nrows = math.ceil(len(names) / cols)
fig, axes = plt.subplots(nrows, cols, figsize=(6.5 * cols, 5.8 * nrows), squeeze=False)
for ax in axes.ravel()[len(names):]:
    ax.axis("off")
for ax, name in zip(axes.ravel(), names):
        p0, d = proj[name]
        ax.scatter(p0[:, 0], p0[:, 1], s=2, color="lightgrey")
        for k in range(K):
            m = pick[labels[pick] == k]
            ax.quiver(p0[m, 0], p0[m, 1], d[m, 0] * scale, d[m, 1] * scale, color=STYLE_CMAP(k),
                      angles="xy", scale_units="xy", scale=1, width=0.004, label=class_names[k])
        ax.set_title(f"{name} (arrows × {scale:.3g})", fontsize=10); ax.set_xticks([]); ax.set_yticks([])
handles, labs = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labs, loc="center right", fontsize=9)
plt.tight_layout(rect=(0, 0, 0.86, 1)); plt.show()

## 7. Do labels and exposure push paintings in the same directions?

For each painting, the cosine similarity between its movement under labels and under exposure (shared shift removed), at the final epoch. Near 0 = unrelated directions; positive = same; negative = opposite.

In [ ]:
pairs = [(b, w) for b in BASES for w in WARMUPS]
found = False
for b, w in pairs:
    nl, ne = f"{short(b)} · labels · warm-up {w}", f"{short(b)} · exposure · warm-up {w}"
    if nl not in MAIN or ne not in MAIN:
        continue
    found = True
    E0 = R["before"][b]["emb"]
    Dl = field_stats(E0, MAIN[nl]["embs"][max(MAIN[nl]["embs"])].astype(np.float32))["Dr"]
    De = field_stats(E0, MAIN[ne]["embs"][max(MAIN[ne]["embs"])].astype(np.float32))["Dr"]
    cos = (Dl * De).sum(1) / (np.linalg.norm(Dl, axis=1) * np.linalg.norm(De, axis=1) + 1e-12)
    by_style = {class_names[k]: cos[labels == k].mean() for k in range(K)}
    print(f"{short(b)}, warm-up {w}: mean cosine {cos.mean():+.3f}  |  by style: " +
          ", ".join(f"{s} {v:+.2f}" for s, v in by_style.items()))
if not found:
    print("Needs both objectives: set OBJECTIVES = ['labels', 'exposure'].")

## 8. Local spatial structure: does label training erode it?

Everything above uses the CLS token, the model's single summary of a painting. Here we look inside each painting, at how its patches relate to each other (the "spatial structure" that recent representation-alignment work found matters more than global accuracy).

- **Structure kept:** how similar each painting's patch-to-patch similarity pattern stays to the untrained model's (1 = unchanged).
- **Local–distant contrast:** how much more alike neighbouring patches are than distant ones (higher = more spatially organised).
- **Patch sameness:** average similarity of all patch pairs (rising = the global summary swamping local detail).

Bottom row: the same measures against amount of global change (1 − CKA), so the objectives are compared at equal change. **The hypothesis holds if labels lose more local structure than exposure at the same amount of global change.**

In [ ]:
sp_panels = [("spatial_preserve", "Local structure kept (1 = unchanged)"),
             ("spatial_local", "Local–distant contrast"), ("spatial_homog", "Patch sameness")]
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for (name, seed), run in R["runs"].items():
    df = TRAJ[(name, seed)]
    kw = dict(color=RUN_COL[name], linestyle=OBJ_LINE[run["objective"]], marker="o", markersize=3,
              alpha=1.0 if seed == SEEDS[0] else 0.3, label=name if seed == SEEDS[0] else None)
    for j, (key, title) in enumerate(sp_panels):
        axes[0, j].plot(df["epoch"], df[key], **kw)
        axes[1, j].plot(1 - df["cka_to_base"], df[key], **kw)
for j, (key, title) in enumerate(sp_panels):
    axes[0, j].set_title(title, fontsize=10); axes[0, j].set_xlabel("epoch")
    axes[1, j].set_title(f"{title} vs amount of global change", fontsize=10)
    axes[1, j].set_xlabel("global change from untrained (1 − CKA)")
for ax in axes.ravel():
    ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.show()

print("Final epoch:")
for name, run in MAIN.items():
    t = TRAJ[(name, SEEDS[0])].iloc[-1]; b = R["before"][run["base"]]["spatial"]
    print(f"  {name}: structure kept {t['spatial_preserve']:.3f}, local contrast {b['spatial_local']:.3f} → "
          f"{t['spatial_local']:.3f}, patch sameness {b['spatial_homog']:.3f} → {t['spatial_homog']:.3f}")

**What one patch "goes with".** For a few test paintings, the similarity of one query patch (white dot; change `QUERY` in the settings) to every other patch, before training and after each objective. Bright = treated as similar to the query. If label training erodes local structure, the "after labels" maps should look flatter or less tied to the painting's shapes.

In [ ]:
rows = [(b, p) for b in BASES for p in range(len(EX_POS))]
run_cols = list(MAIN)
cols = ["painting", "before"] + run_cols
fig, axes = plt.subplots(len(rows), len(cols), figsize=(2.9 * len(cols), 2.9 * len(rows)), squeeze=False)
for i, (b, p) in enumerate(rows):
    img = show_img(images[te_idx[EX_POS[p]]])
    S0e = R["before"][b]["S_examples"][p]
    side = int(math.sqrt(S0e.shape[0]))
    qr, qc = min(int(QUERY[0] * side), side - 1), min(int(QUERY[1] * side), side - 1)
    q = qr * side + qc
    maps = {"before": S0e[q]}
    for rn in run_cols:
        run = MAIN[rn]
        if run["base"] == b and "S_examples" in run:
            maps[rn] = run["S_examples"][p][q]
    for j, col in enumerate(cols):
        ax = axes[i, j]
        ax.imshow(img)
        if col != "painting" and col in maps:
            ax.imshow(upsample(maps[col].reshape(side, side)), cmap="magma", alpha=0.7)
        ax.scatter([(qc + 0.5) * 224 / side], [(qr + 0.5) * 224 / side], s=30, c="white", edgecolors="black")
        ax.set_xticks([]); ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(f"{short(b)}\n{class_names[labels[te_idx[EX_POS[p]]]]}", fontsize=8)
        if i == 0:
            ax.set_title(col.replace(" · ", "\n"), fontsize=8)
plt.tight_layout(); plt.show()

## How to read it

- **Section 0, the dip test:** if exposure without warm-up dips in most seeds but the warm-up runs don't, the dip was caused by the random head. If the warm-up runs also dip, consistently, exposure genuinely moves abstract and representational art closer early in training.

- **Section 1:** does exposure keep changing with longer training, and does the label split keep falling after its early peak (the neural-collapse lead)?
- **Section 2, the key figure:** at the *same* amount of change, if labels raise the split far more than exposure, the difference is about the learning signal, not training length.
- **Sections 4–6:** a large "style" share means training acts as a field organised by style; a large "shared" share means a simple vector. Section 5 shows whether the field pushes abstract and representational art apart or together.
- **Section 7:** whether the two objectives move paintings in related or unrelated directions.
- **Section 8:** whether label training trades local structure for global categories. Compare labels and exposure at the same amount of global change (bottom row).